# Importación y Exportación de Archivos `.arff` en Python (Librería Estándar)

**Curso de Programación de Inteligencia Artificial**

En este cuaderno aprenderás a leer y escribir archivos en formato **ARFF** (*Attribute-Relation File Format*, estándar de Weka) utilizando **únicamente la librería estándar de Python** (`random`, `with open()`, listas, tuplas y diccionarios), sin depender de paquetes externos como `scikit-learn`, `scipy` o `pandas`.

---

## Parte 1: Creación e Importación del Dataset `breast-cancer.arff`

Primero generaremos localmente una muestra fiel del dataset clásico de Weka `breast-cancer.arff` para que puedas ejecutar este cuaderno sin depender de rutas externas. Después, construiremos un analizador (*parser*) propio usando listas y diccionarios.

In [1]:
# Generamos el archivo de ejemplo 'breast-cancer.arff' en el directorio actual
contenido_breast_cancer = """@relation breast-cancer

@attribute age {'10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90-99'}
@attribute menopause {'lt40','ge40','premeno'}
@attribute tumor-size {'0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59'}
@attribute inv-nodes {'0-2','3-5','6-8','9-11','12-14','15-17','18-20','21-23','24-26','27-29','30-32','33-35','36-39'}
@attribute node-caps {'yes','no'}
@attribute deg-malig {'1','2','3'}
@attribute breast {'left','right'}
@attribute breast-quad {'left_up','left_low','right_up','right_low','central'}
@attribute irradiat {'yes','no'}
@attribute Class {'no-recurrence-events','recurrence-events'}

@data
'40-49','premeno','15-19','0-2','yes','3','right','left_up','no','recurrence-events'
'50-59','ge40','15-19','0-2','no','1','right','central','no','no-recurrence-events'
'50-59','ge40','35-39','0-2','no','2','left','left_low','no','recurrence-events'
'40-49','premeno','35-39','0-2','yes','3','right','left_low','yes','no-recurrence-events'
'40-49','premeno','30-34','3-5','yes','2','left','right_up','no','recurrence-events'
'50-59','premeno','25-29','3-5','no','2','right','left_up','yes','no-recurrence-events'
"""

with open("breast-cancer.arff", "w", encoding="utf-8") as f:
    f.write(contenido_breast_cancer)

print("Archivo 'breast-cancer.arff' creado correctamente.")

Archivo 'breast-cancer.arff' creado correctamente.


### Función de lectura manual de archivos `.arff`

A continuación, leemos `breast-cancer.arff` línea por línea y lo estructuramos en un diccionario con tres claves:
- `'relacion'`: cadena con el nombre de la relación (`@relation`).
- `'atributos'`: lista de tuplas `(nombre_atributo, definicion_tipo)`.
- `'datos'`: lista de diccionarios (o lista de listas) con cada registro de `@data`.

In [2]:
def leer_arff(ruta_archivo):
    relacion = ""
    atributos = []  # Lista de tuplas: (nombre, tipo_o_dominio)
    datos = []      # Lista de diccionarios: {atributo: valor}
    leyendo_datos = False

    with open(ruta_archivo, "r", encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()
            # Ignoramos líneas vacías y comentarios (%)
            if not linea or linea.startswith("%"):
                continue

            linea_lower = linea.lower()

            if linea_lower.startswith("@relation"):
                relacion = linea.split(maxsplit=1)[1].strip()

            elif linea_lower.startswith("@attribute"):
                partes = linea.split(maxsplit=2)
                nombre_attr = partes[1].strip("'\"")
                tipo_attr = partes[2].strip()
                atributos.append((nombre_attr, tipo_attr))

            elif linea_lower.startswith("@data"):
                leyendo_datos = True

            elif leyendo_datos:
                # Separamos valores por coma y limpiamos comillas simples si existen
                valores = [v.strip().strip("'\"") for v in linea.split(",")]
                # Creamos un diccionario emparejando nombres de atributos con sus valores
                registro = {atributos[i][0]: valores[i] for i in range(len(atributos))}
                datos.append(registro)

    return {
        "relacion": relacion,
        "atributos": atributos,
        "datos": datos
    }

# Probamos la importación
dataset_cancer = leer_arff("breast-cancer.arff")

print("Relación:", dataset_cancer["relacion"])
print("Total de atributos:", len(dataset_cancer["atributos"]))
print("Total de registros:", len(dataset_cancer["datos"]))
print("\nPrimeros 3 atributos (tuplas):")
for attr in dataset_cancer["atributos"][:3]:
    print(" ", attr)

print("\nPrimer registro como diccionario:")
print(" ", dataset_cancer["datos"][0])

Relación: breast-cancer
Total de atributos: 10
Total de registros: 6

Primeros 3 atributos (tuplas):
  ('age', "{'10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90-99'}")
  ('menopause', "{'lt40','ge40','premeno'}")
  ('tumor-size', "{'0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59'}")

Primer registro como diccionario:
  {'age': '40-49', 'menopause': 'premeno', 'tumor-size': '15-19', 'inv-nodes': '0-2', 'node-caps': 'yes', 'deg-malig': '3', 'breast': 'right', 'breast-quad': 'left_up', 'irradiat': 'no', 'Class': 'recurrence-events'}


--- 
## Parte 2: Simulación y Exportación a `.arff` (Ejemplo: *Pares o Nones*)

Ahora vamos a simular 5 rondas del juego **Pares o Nones** utilizando únicamente estructuras básicas de Python (`list`, `tuple`, `dict`) y el módulo estándar `random` para que la máquina elija un número aleatorio de dedos entre `0` y `5`.

Después, exportaremos el historial al archivo `partidas_pares_nones.arff` y lo volveremos a leer con nuestra función `leer_arff()` para verificar que la sintaxis sea 100% válida.

In [3]:
import random

# Semilla opcional para reproducibilidad del ejemplo
random.seed(42)

# 1. Definimos el esquema de atributos mediante una lista de tuplas (nombre, tipo)
esquema_atributos = [
    ("ronda", "numeric"),
    ("eleccion_usuario", "{pares, nones}"),
    ("dedos_usuario", "numeric"),
    ("dedos_maquina", "numeric"),
    ("ganador", "{usuario, maquina}")
]

# 2. Simulamos las jugadas del usuario (lista de tuplas) y la respuesta aleatoria de la máquina
jugadas_usuario = [
    ("pares", 2),
    ("nones", 1),
    ("pares", 4),
    ("pares", 3),
    ("nones", 2)
]

historial_partidas = []  # Lista de diccionarios con los resultados

for num_ronda, (eleccion, dedos_usr) in enumerate(jugadas_usuario, start=1):
    dedos_maq = random.randint(0, 5)  # Elección aleatoria de la máquina (0 a 5 dedos)
    suma = dedos_usr + dedos_maq
    resultado_paridad = "pares" if suma % 2 == 0 else "nones"
    ganador = "usuario" if eleccion == resultado_paridad else "maquina"

    registro = {
        "ronda": num_ronda,
        "eleccion_usuario": eleccion,
        "dedos_usuario": dedos_usr,
        "dedos_maquina": dedos_maq,
        "ganador": ganador
    }
    historial_partidas.append(registro)

# 3. Función genérica para exportar a formato ARFF estándar
def exportar_a_arff(registros, atributos, archivo_salida, nombre_relacion):
    with open(archivo_salida, "w", encoding="utf-8") as f:
        # Cabecera @relation
        f.write(f"@relation {nombre_relacion}\n\n")

        # Definición de @attribute
        for nombre_attr, tipo_attr in atributos:
            f.write(f"@attribute {nombre_attr} {tipo_attr}\n")

        # Sección @data
        f.write("\n@data\n")
        for fila in registros:
            # Extraemos los valores respetando estrictamente el orden de los atributos
            valores = [str(fila[nombre_attr]).strip() for nombre_attr, _ in atributos]
            f.write(",".join(valores) + "\n")

    print(f"Archivo '{archivo_salida}' generado correctamente.")

# Ejecutamos la exportación
exportar_a_arff(
    registros=historial_partidas,
    atributos=esquema_atributos,
    archivo_salida="partidas_pares_nones.arff",
    nombre_relacion="juego_pares_o_nones"
)

# 4. Mostramos el contenido bruto y verificamos con nuestra función leer_arff()
print("\n--- Contenido bruto del archivo generado ---")
with open("partidas_pares_nones.arff", "r", encoding="utf-8") as f:
    print(f.read().strip())

print("\n--- Verificación releyendo con leer_arff() ---")
datos_verificados = leer_arff("partidas_pares_nones.arff")
for r in datos_verificados["datos"]:
    print(f"Ronda {r['ronda']}: Usuario eligió '{r['eleccion_usuario']}' ({r['dedos_usuario']} vs {r['dedos_maquina']}) -> Ganador: {r['ganador']}")

Archivo 'partidas_pares_nones.arff' generado correctamente.

--- Contenido bruto del archivo generado ---
@relation juego_pares_o_nones

@attribute ronda numeric
@attribute eleccion_usuario {pares, nones}
@attribute dedos_usuario numeric
@attribute dedos_maquina numeric
@attribute ganador {usuario, maquina}

@data
1,pares,2,5,maquina
2,nones,1,0,usuario
3,pares,4,0,usuario
4,pares,3,5,usuario
5,nones,2,2,maquina

--- Verificación releyendo con leer_arff() ---
Ronda 1: Usuario eligió 'pares' (2 vs 5) -> Ganador: maquina
Ronda 2: Usuario eligió 'nones' (1 vs 0) -> Ganador: usuario
Ronda 3: Usuario eligió 'pares' (4 vs 0) -> Ganador: usuario
Ronda 4: Usuario eligió 'pares' (3 vs 5) -> Ganador: usuario
Ronda 5: Usuario eligió 'nones' (2 vs 2) -> Ganador: maquina
